# Amazon Review Intelligence Pipeline — Exploration Notebook

Interactive exploration of the Amazon Beauty reviews dataset,
sentiment scoring results, and divergence analysis output.

## Section 1 — Setup & Imports

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from google.cloud import bigquery
from huggingface_hub import login

load_dotenv(dotenv_path=os.path.join("..", ".env"))

HF_TOKEN = os.getenv("HF_TOKEN")
GCP_PROJECT = os.getenv("GCP_PROJECT")
BQ_DATASET = os.getenv("BQ_DATASET")
BQ_TABLE = os.getenv("BQ_TABLE")

login(token=HF_TOKEN)

client = bigquery.Client(project=GCP_PROJECT)
print("Setup complete.")

## Section 2 — Load & Verify Data

In [ ]:
cache_path = os.path.join("..", "outputs", "raw_reviews_cache.csv")

if os.path.exists(cache_path):
    df = pd.read_csv(cache_path)
    print("Loaded from local cache.")
else:
    print("Cache not found. Run pipeline.py first.")
    df = pd.DataFrame()

if not df.empty:
    print(f"Shape: {df.shape}")
    print(f"\nDtypes:\n{df.dtypes}")
    print(f"\nFirst 5 rows:")
    display(df.head())
    print(f"\nNull counts per column:")
    print(df.isnull().sum())

## Section 3 — EDA

In [ ]:
if not df.empty:
    print("Descriptive statistics (numeric columns):")
    display(df.describe())

    print("\nRating value counts:")
    print(df["rating"].value_counts().sort_index())

    print("\nTop 10 most reviewed parent_asins:")
    top_asins = df["parent_asin"].value_counts().head(10)
    print(top_asins)

## Section 4 — BigQuery Confirmation

In [ ]:
raw_count_query = f"SELECT COUNT(*) AS row_count FROM `{GCP_PROJECT}.{BQ_DATASET}.{BQ_TABLE}`"
raw_count = client.query(raw_count_query).to_dataframe()
print(f"raw_reviews row count: {raw_count['row_count'].iloc[0]:,}")

sentiment_count_query = f"SELECT COUNT(*) AS row_count FROM `{GCP_PROJECT}.{BQ_DATASET}.reviews_with_sentiment`"
sentiment_count = client.query(sentiment_count_query).to_dataframe()
print(f"reviews_with_sentiment row count: {sentiment_count['row_count'].iloc[0]:,}")

## Section 5 — Divergence Results

In [ ]:
divergence_path = os.path.join("..", "outputs", "divergence_results.csv")

if os.path.exists(divergence_path):
    div_df = pd.read_csv(divergence_path)
    print(f"Divergence results shape: {div_df.shape}")

    print("\nTop 20 products by divergence_magnitude:")
    display(div_df.nlargest(20, "divergence_magnitude"))

    print("\nRisk tier distribution:")
    print(div_df["risk_tier"].value_counts())
else:
    print("Divergence results not found. Run divergence_analysis.py first.")
    div_df = pd.DataFrame()

## Section 6 — Visualizations

### Plot 1: Rating Distribution Histogram

In [ ]:
if not df.empty:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(df["rating"], bins=[0.5, 1.5, 2.5, 3.5, 4.5, 5.5],
            edgecolor="black", color="#4C72B0", rwidth=0.85)
    ax.set_xlabel("Rating")
    ax.set_ylabel("Count")
    ax.set_title("Rating Distribution \u2014 Amazon Beauty Reviews")
    ax.set_xticks([1, 2, 3, 4, 5])
    plt.tight_layout()
    plt.show()

### Plot 2: Top 20 Products by Divergence Magnitude

In [ ]:
if not div_df.empty:
    top20 = div_df.nlargest(20, "divergence_magnitude")
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.barh(top20["parent_asin"].astype(str), top20["divergence_magnitude"],
            color="#DD8452", edgecolor="black")
    ax.set_xlabel("Divergence Magnitude")
    ax.set_ylabel("Product ASIN")
    ax.set_title("Products with Highest Sentiment-Rating Divergence")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

### Plot 3: Risk Tier Distribution

In [ ]:
if not div_df.empty:
    risk_counts = div_df["risk_tier"].value_counts()
    colors = {"HIGH RISK": "#C44E52", "MEDIUM RISK": "#F0C05A", "LOW RISK": "#55A868"}
    plot_colors = [colors.get(tier, "#999999") for tier in risk_counts.index]

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.pie(risk_counts.values, labels=risk_counts.index, autopct="%1.1f%%",
           colors=plot_colors, startangle=140, textprops={"fontsize": 12})
    ax.set_title("Product Risk Tier Distribution")
    plt.tight_layout()
    plt.show()